In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
import os

os.environ["USER_AGENT"] = "research-notebook/1.0 (langchain study)"

import wikipedia
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.tools.file_management import WriteFileTool
from langchain_community.utilities import (
    DuckDuckGoSearchAPIWrapper,
    WikipediaAPIWrapper,
)
from langchain_core.tools import tool

wikipedia.set_user_agent(os.environ["USER_AGENT"])


@tool
def wikipedia_search(query: str) -> str:
    """Search Wikipedia for the given query."""
    wiki = WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=3000)
    return wiki.run(query)


@tool
def duckduckgo_search(query: str) -> str:
    """Search DuckDuckGo. Returns title, link and snippet for each result."""
    ddg = DuckDuckGoSearchAPIWrapper()
    results = ddg.results(query, max_results=5)
    return "\n\n".join(f"{r['title']}\n{r['link']}\n{r['snippet']}" for r in results)


@tool
def scrape_website(url: str) -> str:
    """Get the text content of a website."""
    docs = WebBaseLoader(url).load()
    text = "\n".join(
        line.strip() for line in docs[0].page_content.splitlines() if line.strip()
    )
    return text[:8000]


write_file = WriteFileTool()

/var/folders/3y/15zgx_896nsf1dpx6bmzn2180000gn/T/ipykernel_4152/3968248938.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


In [ ]:
from langchain.agents import create_agent
from langchain_openrouter import ChatOpenRouter

llm = ChatOpenRouter(model="openai/gpt-4o-mini", temperature=0.1)

agent = create_agent(
    llm,
    tools=[wikipedia_search, duckduckgo_search, scrape_website, write_file],
    system_prompt="""You are a research assistant.
Search Wikipedia and DuckDuckGo first.
If DuckDuckGo gives you a website, open it with scrape_website and read the content.
Then write up the research and save it to research.txt with write_file.""",
)

In [4]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Research about the XZ backdoor"}]}
)

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Research about the XZ backdoor
================================== Ai Message ==================================
Tool Calls:
  wikipedia_search (call_Dakknz6ABpio9rWtIXoO0Uve)
 Call ID: call_Dakknz6ABpio9rWtIXoO0Uve
  Args:
    query: XZ backdoor
  duckduckgo_search (call_17SF5N4wGD4hL860cofdq5Ju)
 Call ID: call_17SF5N4wGD4hL860cofdq5Ju
  Args:
    query: XZ backdoor
================================= Tool Message =================================
Name: wikipedia_search

Page: XZ Utils backdoor
Summary: On 29 March 2024, a malicious backdoor was discovered in the compression software XZ Utils. The backdoor gives an attacker who possesses a specific private key the ability to remotely execute code on an affected system through OpenSSH, a set of networking utilities. The backdoor was discovered by software developer Andres Freund.
It was later discovered that the exploit was deliberately included in the softw

In [5]:
print(open("research.txt", encoding="utf-8").read())

### XZ Backdoor Overview

The XZ backdoor, identified as CVE-2024-3094, was a malicious code insertion discovered in the XZ Utils data compression library, specifically in versions 5.6.0 and 5.6.1. This backdoor allowed remote code execution on affected systems by manipulating the OpenSSH server's authentication process under certain conditions. The backdoor was inserted by a contributor using the pseudonym "Jia Tan" and was discovered by software developer Andres Freund on March 29, 2024.

### Technical Details

The backdoor exploited build-time modifications to the liblzma library, injecting obfuscated bytecode that altered public-key authentication responses in the OpenSSH daemon (sshd). This was done without triggering typical anomaly detection systems, making it particularly insidious. The malicious code was embedded through a series of complex modifications that evaded peer review and automated testing, highlighting vulnerabilities in the open-source software supply chain.

### D